# Laboratorium 2 – Automatyczna optymalizacja hiperparametrów (Optuna)

**Autor:** Patryk Kaniosz, nr indeksu 99564

Kolejność komórek: ćw. 1 (wersja Optuny) → ćw. 2 (model bazowy) → ćw. 3 (funkcja celu z rozszerzoną
przestrzenią – ćw. samodzielne 1: `min_samples_leaf`, `max_features` przez `suggest_categorical`) →
przegląd metod `suggest` → ćw. 4 (optymalizacja) → ćw. 5 (MedianPruner) → ćw. 6 (wizualizacje) →
ćw. samodzielne 2 i 3 → optymalizacja równoległa + SQLite → porównanie samplerów TPE vs losowy → pytania.

In [1]:
import optuna
print(optuna.__version__)

4.6.0


c:\Users\patry\anaconda3\envs\mlops\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Wczytanie zbioru danych
X, y = load_breast_cancer(return_X_y=True)

# Podział na część treningową (80%) i testową (20%)
# stratify zachowuje proporcje klas w obu częściach
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Model bazowy z domyślnymi hiperparametrami
baseline = RandomForestClassifier(random_state=42)

# Ocena modelu bazowego metodą pięciokrotnej walidacji krzyżowej
baseline_cv = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
)
print(
    f"Baseline, dokładność CV: {baseline_cv.mean():.4f} "
    f"(odch. std. {baseline_cv.std():.4f})"
)

# Trening na całym zbiorze treningowym i ocena na zbiorze testowym
baseline.fit(X_train, y_train)
baseline_test_acc = baseline.score(X_test, y_test)
print(f"Baseline, dokładność na zbiorze testowym: {baseline_test_acc:.4f}")

Baseline, dokładność CV: 0.9538 (odch. std. 0.0235)
Baseline, dokładność na zbiorze testowym: 0.9561


In [3]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),

        # Dwa nowe hiperparametry
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None],
        ),
    }

    model = RandomForestClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
    ).mean()

    return score

In [4]:
def objective_przeglad(trial):
    # Liczba całkowita z przedziału
    n_estimators = trial.suggest_int("n_estimators", 50, 400)

    # Liczba całkowita z zadanym krokiem
    max_depth = trial.suggest_int("max_depth", 2, 20, step=2)

    # Liczba zmiennoprzecinkowa z przedziału
    max_features = trial.suggest_float("max_features", 0.5, 1.0)

    # Liczba zmiennoprzecinkowa próbkowana w skali logarytmicznej
    min_impurity_decrease = trial.suggest_float(
        "min_impurity_decrease",
        1e-4,
        1e-1,
        log=True,
    )

    # Wybór kategoryczny z listy wartości
    criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        min_impurity_decrease=min_impurity_decrease,
        criterion=criterion,
        random_state=42,
        n_jobs=-1,
    )

    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
    ).mean()

    return score

In [5]:
# Utworzenie badania — dokładność jest maksymalizowana
study = optuna.create_study(
    direction="maximize",
    study_name="rf_tuning",
)

# Uruchomienie optymalizacji obejmującej 50 prób
study.optimize(objective, n_trials=50)

print("Najlepsza dokładność CV:", study.best_value)
print("Najlepsza konfiguracja hiperparametrów:", study.best_params)

[I 2026-10-16 12:58:28,793] A new study created in memory with name: rf_tuning
[I 2026-10-16 12:58:36,437] Trial 0 finished with value: 0.9538461538461538 and parameters: {'n_estimators': 352, 'max_depth': 19, 'min_samples_split': 9, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 12:58:41,772] Trial 1 finished with value: 0.9494505494505494 and parameters: {'n_estimators': 228, 'max_depth': 8, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 12:58:49,578] Trial 2 finished with value: 0.9538461538461538 and parameters: {'n_estimators': 320, 'max_depth': 11, 'min_samples_split': 14, 'min_samples_leaf': 9, 'max_features': None}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 12:58:59,309] Trial 3 finished with value: 0.9516483516483516 and parameters: {'n_estimators': 391, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_l

Najlepsza dokładność CV: 0.9604395604395606
Najlepsza konfiguracja hiperparametrów: {'n_estimators': 278, 'max_depth': 14, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None}


In [6]:
# Utworzenie najlepszego modelu na podstawie parametrów znalezionych przez Optunę
best_model = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
)

# Trening i ocena na zbiorze testowym
best_model.fit(X_train, y_train)
tuned_test_acc = best_model.score(X_test, y_test)

print(f"Model strojony, dokładność na zbiorze testowym: {tuned_test_acc:.4f}")
print(f"Model bazowy, dokładność na zbiorze testowym: {baseline_test_acc:.4f}")

Model strojony, dokładność na zbiorze testowym: 0.9474
Model bazowy, dokładność na zbiorze testowym: 0.9561


In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from optuna.trial import TrialState

# Wydzielenie zbioru walidacyjnego z części treningowej
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train,
)


def objective_pruning(trial):
    params = {
        # Pruner decyduje, czy przerwać próbę przed końcem 300 etapów
        "n_estimators": 300,
        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-3,
            0.3,
            log=True,
        ),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
    }

    model = GradientBoostingClassifier(
        **params,
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    # Ocena po kolejnych etapach uczenia
    for step, y_pred in enumerate(model.staged_predict(X_val)):
        accuracy = accuracy_score(y_val, y_pred)
        trial.report(accuracy, step)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return accuracy


study_pruned = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,  # pierwsze 5 prób nie jest przycinanych
        n_warmup_steps=20,   # przycinanie możliwe od kroku 20
    ),
)

study_pruned.optimize(objective_pruning, n_trials=40)

# Podsumowanie wyników
pruned = [
    trial
    for trial in study_pruned.trials
    if trial.state == TrialState.PRUNED
]
complete = [
    trial
    for trial in study_pruned.trials
    if trial.state == TrialState.COMPLETE
]

print(f"Próby ukończone: {len(complete)}, próby przycięte: {len(pruned)}")

[I 2026-10-16 13:03:34,352] A new study created in memory with name: no-name-69210c26-6242-414d-8067-522a693f5bfc
[I 2026-10-16 13:03:37,400] Trial 0 finished with value: 0.9649122807017544 and parameters: {'learning_rate': 0.0440525396336501, 'max_depth': 5}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-10-16 13:03:40,153] Trial 1 finished with value: 0.956140350877193 and parameters: {'learning_rate': 0.0023529268037145497, 'max_depth': 8}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-10-16 13:03:43,169] Trial 2 finished with value: 0.9473684210526315 and parameters: {'learning_rate': 0.001466593966030913, 'max_depth': 5}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-10-16 13:03:45,822] Trial 3 finished with value: 0.956140350877193 and parameters: {'learning_rate': 0.020248059917542537, 'max_depth': 6}. Best is trial 0 with value: 0.9649122807017544.
[I 2026-10-16 13:03:48,658] Trial 4 finished with value: 0.9649122807017544 and parameters: {'learnin

Próby ukończone: 15, próby przycięte: 25


In [8]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
)

# Historia optymalizacji
fig1 = plot_optimization_history(study)
fig1.show()

# Ważność hiperparametrów
fig2 = plot_param_importances(study)
fig2.show()

In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Optymalizacja co najmniej 50 prób z rozszerzoną funkcją objective
study_extended = optuna.create_study(
    direction="maximize",
    study_name="rf_tuning_extended",
)
study_extended.optimize(objective, n_trials=50)

# Model bazowy: średnia dokładność z 5-krotnej walidacji krzyżowej
baseline_model = RandomForestClassifier(random_state=42, n_jobs=-1)
baseline_cv_accuracy = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
).mean()

# Dokładność testowa modelu bazowego
baseline_model.fit(X_train, y_train)
baseline_test_accuracy = baseline_model.score(X_test, y_test)

# Model strojony: parametry z najlepszego triala
tuned_model = RandomForestClassifier(
    **study_extended.best_params,
    random_state=42,
    n_jobs=-1,
)
tuned_model.fit(X_train, y_train)
tuned_test_accuracy = tuned_model.score(X_test, y_test)

# Tabela porównawcza
print(f"{'Metryka':<35} {'Bazowy':>12} {'Strojony':>12}")
print("-" * 61)
print(
    f"{'Dokładność walidacji krzyżowej':<35} "
    f"{baseline_cv_accuracy:>12.4f} {study_extended.best_value:>12.4f}"
)
print(
    f"{'Dokładność testowa':<35} "
    f"{baseline_test_accuracy:>12.4f} {tuned_test_accuracy:>12.4f}"
)

[I 2026-10-16 13:05:09,580] A new study created in memory with name: rf_tuning_extended
[I 2026-10-16 13:05:15,295] Trial 0 finished with value: 0.9538461538461538 and parameters: {'n_estimators': 277, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 13:05:19,243] Trial 1 finished with value: 0.9516483516483518 and parameters: {'n_estimators': 155, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 13:05:24,147] Trial 2 finished with value: 0.9494505494505494 and parameters: {'n_estimators': 210, 'max_depth': 11, 'min_samples_split': 19, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.9538461538461538.
[I 2026-10-16 13:05:31,874] Trial 3 finished with value: 0.9472527472527472 and parameters: {'n_estimators': 304, 'max_depth': 2, 'min_samples_split': 11, 'mi

Metryka                                   Bazowy     Strojony
-------------------------------------------------------------
Dokładność walidacji krzyżowej            0.9538       0.9604
Dokładność testowa                        0.9561       0.9474


In [10]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
)

# Historia optymalizacji
plot_optimization_history(study_extended).show()

# Ważność hiperparametrów
plot_param_importances(study_extended).show()

## Ćwiczenie samodzielne 2 – komentarz do porównania

| Model | Dokładność CV | Dokładność testowa |
|---|---|---|
| Bazowy (`RandomForestClassifier` domyślny) | 0.9538 | 0.9561 |
| Strojony (Optuna, 50 prób) | 0.9604 | 0.9474 |

**Czy strojenie przyniosło poprawę?** Na walidacji krzyżowej tak – wzrost o ok. 0.7 p.p.
Na zbiorze testowym nie: model strojony wypadł o ok. 0.9 p.p. gorzej, co przy 114 obserwacjach
testowych oznacza różnicę jednej próbki. Różnica mieści się w odchyleniu standardowym CV (≈0.024),
więc oba modele należy uznać za praktycznie równoważne.

**Czy widać nadmierne dopasowanie do walidacji?** Tak, w niewielkim stopniu. `study.best_value`
jest maksimum z 50 ocen obarczonych szumem, więc jest optymistycznie obciążone – wybraliśmy konfigurację,
która "miała szczęście" na foldach CV. Spadek wyniku na teście względem CV to typowy objaw
dopasowania doboru hiperparametrów do walidacji. Na tak małym i łatwym zbiorze domyślny las losowy
jest już blisko optimum, więc zysk ze strojenia jest mniejszy niż szum pomiaru.

## Ćwiczenie samodzielne 3 – interpretacja wizualizacji

**Historia optymalizacji.** Wykres pokazuje wartość funkcji celu (dokładność CV) w kolejnych próbach
oraz linię najlepszego dotychczasowego wyniku. Linia najlepszego wyniku rośnie skokowo w pierwszych
kilkunastu próbach, a od ok. 30. próby (najlepsza: 0.9604) jest płaska – dalsze zwiększanie budżetu
prawdopodobnie nie przyniosłoby istotnej poprawy, 50 prób to wystarczający budżet dla tego zbioru.

**Ważność hiperparametrów.** Wykres szacuje (metodą fANOVA) udział każdego hiperparametru w zmienności
funkcji celu. Dominuje `min_samples_split` (≈0.50), dalej `max_features` (≈0.19) i `min_samples_leaf`
(≈0.16), a `n_estimators` (≈0.08) i `max_depth` (≈0.07) mają znikomy wpływ. Praktyczny wniosek:
w kolejnych eksperymentach można ustalić `n_estimators` i `max_depth` na stałym poziomie
i skupić budżet na parametrach kontrolujących podział węzłów oraz losowanie cech.

## Optymalizacja równoległa i trwałe przechowywanie wyników w SQLite

Badanie zapisywane jest w pliku `optuna.db`. Dzięki `load_if_exists=True` można je wznowić
(albo uruchomić ten sam kod w kilku procesach/terminalach, które współdzielą jedno badanie).
`n_jobs=4` uruchamia 4 próby równolegle w wątkach jednego procesu.

In [ ]:
storage = "sqlite:///optuna.db"

study_db = optuna.create_study(
    direction="maximize",
    study_name="rf_tuning_sqlite",
    storage=storage,
    load_if_exists=True,
)
study_db.optimize(objective, n_trials=40, n_jobs=4)

print("Liczba prob w bazie:", len(study_db.trials))
print("Najlepsza dokladnosc CV:", study_db.best_value)

# Wczytanie badania z bazy (np. w innym procesie lub po restarcie kernela)
loaded = optuna.load_study(study_name="rf_tuning_sqlite", storage=storage)
loaded.trials_dataframe().sort_values("value", ascending=False).head()

Podgląd w przeglądarce (opcjonalnie): `pip install optuna-dashboard`, następnie
`optuna-dashboard sqlite:///optuna.db`.

## Porównanie samplerów TPE i losowego

Oba badania mają ten sam budżet (50 prób) i to samo ziarno. Porównujemy najlepszy wynik
oraz to, jak szybko każdy sampler go osiąga.

In [ ]:
import pandas as pd
from optuna.samplers import RandomSampler, TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 50
samplers = {
    "TPE": TPESampler(seed=42),
    "Random": RandomSampler(seed=42),
}

wyniki = {}
historie = {}
for nazwa, sampler in samplers.items():
    s = optuna.create_study(direction="maximize", sampler=sampler)
    s.optimize(objective, n_trials=N_TRIALS)
    values = [t.value for t in s.trials]
    best_so_far = pd.Series(values).cummax()
    historie[nazwa] = best_so_far
    wyniki[nazwa] = {
        "best_cv": s.best_value,
        "proba_najlepsza": s.best_trial.number,
        "srednia_ostatnich_20": pd.Series(values[-20:]).mean(),
    }

pd.DataFrame(wyniki).T

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for nazwa, seria in historie.items():
    fig.add_trace(go.Scatter(y=seria, mode="lines", name=nazwa))
fig.update_layout(
    title="Najlepszy dotychczasowy wynik: TPE vs Random",
    xaxis_title="Numer proby",
    yaxis_title="Dokladnosc CV",
)
fig.show()

**Wniosek:** TPE po fazie startowej (domyślnie 10 losowych prób) zaczyna proponować
konfiguracje z obiecujących rejonów przestrzeni, dlatego średnia ostatnich prób jest u niego
zwykle wyższa niż dla samplera losowego, a najlepszy wynik osiągany jest szybciej.
Na małym i łatwym zbiorze różnica najlepszego wyniku bywa niewielka (rzędu 0.002–0.005),
bo wiele konfiguracji daje zbliżoną dokładność. *(Po uruchomieniu uzupełnić o własne liczby z tabeli.)*

## Pytania pomocnicze

1. **Parametr vs hiperparametr.** Parametry są wyznaczane z danych podczas uczenia (np. wagi regresji
   liniowej, progi podziałów w drzewie). Hiperparametry ustala projektant przed treningiem
   (np. liczba drzew `n_estimators`, współczynnik uczenia `learning_rate`).
2. **Grid search.** Liczba kombinacji rośnie wykładniczo z liczbą hiperparametrów (10^k).
   Dla 5 hiperparametrów po 10 wartości to 10^5 = **100 000** treningów (×5 przy 5-krotnej CV).
3. **Define-by-run.** Przestrzeń powstaje dynamicznie podczas wykonywania funkcji celu (wywołania
   `trial.suggest_*`), a nie jako statyczny słownik. Pozwala to tworzyć przestrzenie warunkowe,
   np. inne hiperparametry dla SVM, a inne dla lasu losowego, zależnie od wcześniejszego wyboru.
4. **Skala logarytmiczna.** Gdy sensowny zakres obejmuje kilka rzędów wielkości i istotny jest
   stosunek, a nie różnica wartości. Np. `learning_rate` w zakresie 1e-4–1e-1: zmiana z 0.0001 na 0.001
   jest tak samo istotna jak z 0.01 na 0.1; próbkowanie liniowe marnowałoby ~90% prób na zakres 0.01–0.1.
5. **CV zamiast testu.** Zbiór testowy ma dawać nieobciążoną ocenę modelu na nowych danych.
   Gdyby funkcja celu liczona była na teście, hiperparametry zostałyby do niego dopasowane (wyciek
   informacji) – raportowana jakość byłaby zawyżona, a nie mielibyśmy już żadnych "świeżych"
   danych do uczciwej oceny.